# TeamDetector bootstrap: con filtro vs sin filtro

Este notebook recrea los dos escenarios sobre `data/partidoPrueba/partido_corto.mp4`.

- Experimento 1: con filtro por campo proyectado.
- Experimento 2: sin ese filtro.

En cada experimento se muestran las primeras 60 muestras de `shirt_color` que llegan al bootstrap del `TeamDetector`, junto con:
- la etiqueta de equipo usando los colores de referencia de `config.yaml`
- el cluster asignado por el KMeans global de actualización

Nota: el código productivo actual dispara el update cuando entra la muestra 61 (`len(...) > min_samples`). Aquí se muestran las primeras 60 muestras pedidas para inspección visual, y el clustering se calcula exactamente sobre esas 60.

In [ ]:
from __future__ import annotations

import math
import sys
import types
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    def display(value):
        print(value)

    def Markdown(value):
        return value
from sklearn.cluster import KMeans

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config.yaml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No se encontró config.yaml al ascender desde el directorio actual.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from football_ai.core import get_config
from football_ai.tracking import Tracker

VIDEO_PATH = PROJECT_ROOT / "data/partidoPrueba/partido_corto.mp4"
SAMPLE_LIMIT = 60
GRID_COLUMNS = 10
CLUSTER_MARKERS = {0: "o", 1: "s"}

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"


def lab_opencv_to_rgb01(lab_color: np.ndarray) -> tuple[float, float, float]:
    lab_pixel = np.array([[lab_color]], dtype=np.float32)
    lab_pixel_8u = np.clip(np.round(lab_pixel), 0, 255).astype(np.uint8)
    rgb_pixel = cv2.cvtColor(lab_pixel_8u, cv2.COLOR_LAB2RGB)[0, 0]
    return tuple((rgb_pixel.astype(np.float32) / 255.0).tolist())


def build_tracker(disable_field_filter: bool = False) -> Tracker:
    config = get_config()
    detector_conf = dict(config.detection)
    detector_conf["verbose"] = False

    tracker = Tracker(
        model_path=str(config.get_path("paths", "models", "modelo_base")),
        detector_conf=detector_conf,
        team_detector_conf=dict(config.team_detector),
        bytetracker_conf=config.bytetracker,
        ball_conf=config.ball,
        tracker_conf=config.tracking,
        projector_conf=config.projector,
        project_root=config.project_root,
    )

    if disable_field_filter:
        def no_filter(
            self,
            detections,
            detections_sv,
            raw_detections,
            detection_class_labels,
            field_projection,
            field_positions,
            ground_points_projected,
        ):
            return (
                detections,
                detections_sv,
                raw_detections,
                detection_class_labels,
                field_positions,
                ground_points_projected,
            )

        tracker._filter_detections_outside_projected_field = types.MethodType(
            no_filter,
            tracker,
        )

    return tracker


def collect_bootstrap_samples(
    disable_field_filter: bool = False,
    sample_limit: int = SAMPLE_LIMIT,
):
    tracker = build_tracker(disable_field_filter=disable_field_filter)
    team_detector = tracker.team_detector
    samples = []

    for frame_idx, detections in enumerate(tracker.model.detect(str(VIDEO_PATH))):
        tracker._phase_prepare_frame_inputs(detections, collect_visual_debug=False)

        for det_idx, detection in enumerate(detections):
            class_name = detection.names[detection.boxes.cls.item()]
            sample_bucket = "player" if class_name == "goalkeeper" else class_name
            if sample_bucket != "player":
                continue

            shirt_img_bgr, shirt_color = team_detector._get_shirt_color(detection)
            if shirt_img_bgr is None or shirt_color is None or shirt_img_bgr.size == 0:
                continue

            ref_team, distances = team_detector._assign_team(shirt_color)
            samples.append(
                {
                    "sample_index": len(samples),
                    "frame_idx": frame_idx,
                    "det_idx": det_idx,
                    "shirt_img_rgb": cv2.cvtColor(shirt_img_bgr, cv2.COLOR_BGR2RGB),
                    "shirt_color_lab": np.asarray(shirt_color, dtype=np.float32),
                    "shirt_color_rgb01": lab_opencv_to_rgb01(np.asarray(shirt_color, dtype=np.float32)),
                    "ref_team": ref_team,
                    "distances": distances,
                }
            )

            if len(samples) >= sample_limit:
                return tracker, samples

    return tracker, samples


def assign_clusters_like_team_detector(tracker: Tracker, samples: list[dict]):
    team_detector = tracker.team_detector
    sample_matrix = np.asarray(
        [sample["shirt_color_lab"] for sample in samples],
        dtype=np.float32,
    )

    km = KMeans(
        n_clusters=team_detector.n_teams,
        init="k-means++",
        n_init=5,
        random_state=0,
    )
    km.fit(sample_matrix)

    km_checked, samples_checked = team_detector._check_clusters_sizes(km, sample_matrix)
    if km_checked is None or len(samples_checked) != len(sample_matrix):
        labels = np.asarray(km.labels_, dtype=int)
    else:
        labels = np.asarray(km_checked.labels_, dtype=int)
    centers = []
    for cluster_id in range(team_detector.n_teams):
        cluster_samples = sample_matrix[labels == cluster_id]
        centers.append(np.median(cluster_samples, axis=0).astype(np.float32))

    for sample, cluster_id in zip(samples, labels):
        sample["cluster_id"] = int(cluster_id)

    return labels, np.asarray(centers, dtype=np.float32)


def build_summary_table(samples: list[dict]) -> pd.DataFrame:
    table = pd.crosstab(
        pd.Series([sample["cluster_id"] for sample in samples], name="cluster_id"),
        pd.Series([sample["ref_team"] for sample in samples], name="equipo_ref"),
        dropna=False,
    )
    return table


def plot_shirt_grid(samples: list[dict], title: str, columns: int = GRID_COLUMNS):
    ordered_samples = sorted(
        samples,
        key=lambda sample: (
            sample["cluster_id"],
            str(sample["ref_team"]),
            sample["sample_index"],
        ),
    )

    rows = math.ceil(len(ordered_samples) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 1.7, rows * 2.2))
    axes = np.atleast_1d(axes).reshape(rows, columns)

    for axis in axes.ravel():
        axis.axis("off")

    for axis, sample in zip(axes.ravel(), ordered_samples):
        axis.imshow(sample["shirt_img_rgb"])
        axis.set_title(
            f"#{sample['sample_index']:02d} | f{sample['frame_idx']}\n"
            f"ref={sample['ref_team']}\n"
            f"cluster={sample['cluster_id']}",
            fontsize=7,
        )
        for spine in axis.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(1.5)
            spine.set_edgecolor("black" if sample["cluster_id"] == 0 else "dimgray")

    fig.suptitle(title, fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()


def plot_clusters_3d(samples: list[dict], centers: np.ndarray, title: str, tracker: Tracker):
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    for cluster_id in sorted({sample['cluster_id'] for sample in samples}):
        cluster_samples = [sample for sample in samples if sample["cluster_id"] == cluster_id]
        sample_matrix = np.asarray([sample["shirt_color_lab"] for sample in cluster_samples], dtype=np.float32)
        point_colors = [sample["shirt_color_rgb01"] for sample in cluster_samples]

        ax.scatter(
            sample_matrix[:, 0],
            sample_matrix[:, 1],
            sample_matrix[:, 2],
            c=point_colors,
            marker=CLUSTER_MARKERS.get(cluster_id, "o"),
            s=55,
            edgecolors="black",
            linewidths=0.4,
            alpha=0.95,
            label=f"Cluster {cluster_id}",
        )

    for cluster_id, center in enumerate(centers):
        ax.scatter(
            center[0],
            center[1],
            center[2],
            c=[lab_opencv_to_rgb01(center)],
            marker="X",
            s=260,
            edgecolors="black",
            linewidths=1.0,
            label=f"Centro {cluster_id}",
        )

    reference_colors = {
        team_name: np.asarray(color, dtype=np.float32)
        for team_name, color in tracker.team_detector.team_colors.items()
        if team_name != "referee" and color is not None
    }
    for team_name, reference_color in reference_colors.items():
        ax.scatter(
            reference_color[0],
            reference_color[1],
            reference_color[2],
            c=[lab_opencv_to_rgb01(reference_color)],
            marker="^",
            s=220,
            edgecolors="black",
            linewidths=1.0,
            label=f"Ref config: {team_name}",
        )

    ax.set_xlabel("L")
    ax.set_ylabel("a")
    ax.set_zlabel("b")
    ax.set_title(title)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()


def run_experiment(title: str, disable_field_filter: bool = False):
    tracker, samples = collect_bootstrap_samples(
        disable_field_filter=disable_field_filter,
        sample_limit=SAMPLE_LIMIT,
    )
    labels, centers = assign_clusters_like_team_detector(tracker, samples)

    display(Markdown(f"## {title}"))
    display(
        Markdown(
            f"- Video: `{VIDEO_PATH.relative_to(PROJECT_ROOT)}`\n"
            f"- Muestras mostradas: `{len(samples)}`\n"
            f"- Recuento por equipo de referencia: `{dict(Counter(sample['ref_team'] for sample in samples))}`\n"
            f"- Recuento por cluster: `{dict(Counter(labels.tolist()))}`"
        )
    )
    display(build_summary_table(samples))

    plot_shirt_grid(
        samples,
        title=f"{title}: primeras {len(samples)} shirts ordenadas por cluster",
    )
    plot_clusters_3d(
        samples,
        centers,
        title=f"{title}: distribución LAB, centros y referencias config",
        tracker=tracker,
    )

    return {
        "tracker": tracker,
        "samples": samples,
        "labels": labels,
        "centers": centers,
    }


In [ ]:
experiment_1 = run_experiment(
    title="Experimento 1: con filtro por campo proyectado",
    disable_field_filter=False,
)


In [ ]:
experiment_2 = run_experiment(
    title="Experimento 2: sin filtro por campo proyectado",
    disable_field_filter=True,
)
